In [1]:
import numpy as np
mat_size=32
input_width=8
result_width=32

In [2]:
# 32x32 행렬 2개 생성 (8비트 부호 있는 정수: -128 ~ 127)
# np.random.seed(42)  # 재현성을 위한 시드 설정
max_num=2**(input_width-1)
if (input_width==8):
    d_type=np.int8
    h_type=np.uint8 # uint8로 수정

matrix_a = np.random.randint(-max_num, max_num, size=(mat_size, mat_size), dtype=d_type)
vector_b = np.random.randint(-max_num, max_num, size=(mat_size,), dtype=d_type)

# 8비트 2의 보수 형식으로 변환 (0x00 ~ 0xFF)
matrix_a_hex = matrix_a.astype(h_type)
vector_b_hex = vector_b.astype(h_type)


In [3]:
# matrix_a에서 절반을 랜덤으로 0으로 만들어 sparse_a 생성
sparse_a = matrix_a.copy()

# 전체 인덱스 중 절반을 랜덤 선택
total_elements = mat_size * mat_size
indices = np.random.choice(total_elements, size=total_elements // 2, replace=False)

# 선택된 인덱스를 0으로 설정
rows = indices // mat_size
cols = indices % mat_size
sparse_a[rows, cols] = 0

# sparse_a 정보 출력
sparse_a_hex = sparse_a.astype(h_type)

print(f"=== Sparse Matrix A ===")
print(f"Total elements: {sparse_a.size}")
print(f"Non-zero elements: {np.count_nonzero(sparse_a)}")
print(f"Sparsity: {100 * (1 - np.count_nonzero(sparse_a) / sparse_a.size):.2f}%")

print(f"\nSparse A (top-left 4x4):")
print(sparse_a[:4, :4])

print(f"\nSparse A (top-left 4x4, Hex):")
for row in sparse_a_hex[:4, :4]:
    print(" ".join(f"{val:02X}" for val in row))

=== Sparse Matrix A ===
Total elements: 1024
Non-zero elements: 510
Sparsity: 50.20%

Sparse A (top-left 4x4):
[[   0    0    0 -112]
 [   2    0    0    0]
 [   0  -43    0 -107]
 [   0  -92  117  -92]]

Sparse A (top-left 4x4, Hex):
00 00 00 90
02 00 00 00
00 D5 00 95
00 A4 75 A4


In [4]:

def matrix_to_csr(matrix):
    """
    행렬을 CSR (Compressed Sparse Row) 포맷으로 변환
    
    Parameters:
        matrix: 입력 행렬 (2D numpy array)
    
    Returns:
        values: 0이 아닌 값들의 배열
        col_indices: 각 값의 열 인덱스
        row_ptr: 각 행의 시작 위치 (길이 = 행 수 + 1)
    """
    rows, cols = matrix.shape
    values = []
    col_indices = []
    row_ptr = [0]
    
    for i in range(rows):
        for j in range(cols):
            if matrix[i, j] != 0:
                values.append(int(matrix[i, j]))
                col_indices.append(j)
        row_ptr.append(len(values))
    
    return np.array(values), np.array(col_indices), np.array(row_ptr)

# sparse_a를 CSR 포맷으로 변환
csr_values, csr_col_indices, csr_row_ptr = matrix_to_csr(sparse_a)

print(f"=== Sparse A CSR Format ===")
print(f"Original shape: {sparse_a.shape}")
print(f"Total elements: {sparse_a.size}")
print(f"Non-zero elements: {len(csr_values)}")
print(f"Sparsity: {100 * (1 - len(csr_values) / sparse_a.size):.2f}%")

print(f"\nValues (first 20): {csr_values[:20]}")
print(f"Col indices (first 20): {csr_col_indices[:20]}")
print(f"Row pointers (first 10): {csr_row_ptr[:10]}")

# 16진수로 출력
print(f"\nValues (Hex, first 20):")
print(" ".join(f"{v & 0xFF:02X}" for v in csr_values[:20]))

print(f"\nCol indices (Hex, first 20):")
print(" ".join(f"{c:02X}" for c in csr_col_indices[:20]))

print(f"\nRow pointers (Hex, first 10):")
print(" ".join(f"{r:04X}" for r in csr_row_ptr[:10]))

# 검증: CSR에서 원본 행렬 복원
def csr_to_matrix(values, col_indices, row_ptr, shape):
    """CSR 포맷에서 원본 행렬 복원"""
    matrix = np.zeros(shape, dtype=np.int8)
    for i in range(len(row_ptr) - 1):
        for j in range(row_ptr[i], row_ptr[i + 1]):
            matrix[i, col_indices[j]] = values[j]
    return matrix

# 복원 검증 - sparse_a와 비교해야 함!
restored_a = csr_to_matrix(csr_values, csr_col_indices, csr_row_ptr, sparse_a.shape)
print(f"\nReconstruction check: {np.array_equal(sparse_a, restored_a)}")

=== Sparse A CSR Format ===
Original shape: (32, 32)
Total elements: 1024
Non-zero elements: 510
Sparsity: 50.20%

Values (first 20): [-112  -59  -52   -9 -120  -66  -54   28   97   42   91 -105   12   23
    2  -11  -27  -91  -24 -126]
Col indices (first 20): [ 3  4  6 11 12 14 16 18 19 20 21 24 26 29  0  4  6  9 11 12]
Row pointers (first 10): [  0  14  31  53  70  86 104 126 142 159]

Values (Hex, first 20):
90 C5 CC F7 88 BE CA 1C 61 2A 5B 97 0C 17 02 F5 E5 A5 E8 82

Col indices (Hex, first 20):
03 04 06 0B 0C 0E 10 12 13 14 15 18 1A 1D 00 04 06 09 0B 0C

Row pointers (Hex, first 10):
0000 000E 001F 0035 0046 0056 0068 007E 008E 009F

Reconstruction check: True


In [5]:

def save_csr_to_coe(csr_values, csr_col_indices, csr_row_ptr, prefix="csr"):
    """
    CSR 포맷의 각 배열을 COE 파일로 저장
    
    Parameters:
        csr_values: 0이 아닌 값들 (8비트 부호 있는 정수)
        csr_col_indices: 열 인덱스 (5비트, 0~31)
        csr_row_ptr: 행 포인터 (10비트, 0~1023)
        prefix: 파일명 접두어
    """
    
    # 1. Values COE (8비트)
    with open(f"{prefix}_values.coe", 'w') as f:
        f.write("memory_initialization_radix=16;\n")
        f.write("memory_initialization_vector=\n")
        for i, val in enumerate(csr_values):
            hex_val = val & 0xFF  # 8비트 2의 보수
            if i < len(csr_values) - 1:
                f.write(f"{hex_val:02X},\n")
            else:
                f.write(f"{hex_val:02X};\n")
    print(f"{prefix}_values.coe 생성 완료! ({len(csr_values)}줄)")
    
    # 2. Column Indices COE (8비트로 저장, 실제 5비트 사용)
    with open(f"{prefix}_col_indices.coe", 'w') as f:
        f.write("memory_initialization_radix=16;\n")
        f.write("memory_initialization_vector=\n")
        for i, col in enumerate(csr_col_indices):
            if i < len(csr_col_indices) - 1:
                f.write(f"{col:02X},\n")
            else:
                f.write(f"{col:02X};\n")
    print(f"{prefix}_col_indices.coe 생성 완료! ({len(csr_col_indices)}줄)")
    
    # 3. Row Pointers COE (16비트로 저장)
    with open(f"{prefix}_row_ptr.coe", 'w') as f:
        f.write("memory_initialization_radix=16;\n")
        f.write("memory_initialization_vector=\n")
        for i, ptr in enumerate(csr_row_ptr):
            if i < len(csr_row_ptr) - 1:
                f.write(f"{ptr:04X},\n")
            else:
                f.write(f"{ptr:04X};\n")
    print(f"{prefix}_row_ptr.coe 생성 완료! ({len(csr_row_ptr)}줄)")

# CSR 포맷을 COE 파일로 저장
save_csr_to_coe(csr_values, csr_col_indices, csr_row_ptr, "sparse_a_csr")

# 검증 출력
print("\n=== CSR COE 파일 검증 ===")
print(f"Values (first 10): {[f'{v & 0xFF:02X}' for v in csr_values[:10]]}")
print(f"Col indices (first 10): {[f'{c:02X}' for c in csr_col_indices[:10]]}")
print(f"Row pointers (first 5): {[f'{r:04X}' for r in csr_row_ptr[:5]]}")

sparse_a_csr_values.coe 생성 완료! (510줄)
sparse_a_csr_col_indices.coe 생성 완료! (510줄)
sparse_a_csr_row_ptr.coe 생성 완료! (33줄)

=== CSR COE 파일 검증 ===
Values (first 10): ['90', 'C5', 'CC', 'F7', '88', 'BE', 'CA', '1C', '61', '2A']
Col indices (first 10): ['03', '04', '06', '0B', '0C', '0E', '10', '12', '13', '14']
Row pointers (first 5): ['0000', '000E', '001F', '0035', '0046']


In [6]:
print(f"\nSparse A (top-left 4x4):")
print(restored_a[:4, :4])
restored_a_hex = restored_a.astype(h_type)

print(f"\nSparse A (top-left 4x4, Hex):")
for row in restored_a_hex[:4, :4]:
    print(" ".join(f"{val:02X}" for val in row))


Sparse A (top-left 4x4):
[[   0    0    0 -112]
 [   2    0    0    0]
 [   0  -43    0 -107]
 [   0  -92  117  -92]]

Sparse A (top-left 4x4, Hex):
00 00 00 90
02 00 00 00
00 D5 00 95
00 A4 75 A4


In [7]:
# 행렬 A와 B의 좌측 상위 4x4 출력
print("Matrix A (top-left 4x4):")
print(matrix_a[:4, :4])

# 16진수로도 출력
print("\nMatrix A (top-left 4x4, Hex):")
for row in matrix_a_hex[:4, :4]:
    print(" ".join(f"{val:02X}" for val in row))

print(vector_b)

Matrix A (top-left 4x4):
[[  91  106   49 -112]
 [   2   41  -85 -120]
 [  28  -43  -10 -107]
 [  57  -92  117  -92]]

Matrix A (top-left 4x4, Hex):
5B 6A 31 90
02 29 AB 88
1C D5 F6 95
39 A4 75 A4
[  93  -57   38  -50   71   62  124  -76    5  127   90   41  -49 -107
  111  -56   66   79  -44  114   52   31  -82   21   -7   21   -6  -73
   77  -68 -117   37]


In [8]:

def save_matrix_to_coe(matrix, filename, input_width=8):
    """
    행렬을 COE 파일로 저장 (2개 원소를 16비트로 결합)
    첫번째 원소 -> 하위 8비트, 두번째 원소 -> 상위 8비트
    
    Parameters:
        matrix: 저장할 행렬 (int8)
        filename: 출력 파일명
        input_width: 원소당 비트 수 (기본 8비트)
    """
    # uint8로 변환 (2의 보수 유지)
    matrix_hex = matrix.astype(np.uint8)
    rows, cols = matrix.shape
    
    with open(filename, 'w') as f:
        f.write("memory_initialization_radix=16;\n")
        f.write("memory_initialization_vector=\n")
        total_lines = (rows * cols) // 2
        line_idx = 0
        
        for i in range(rows):
            for j in range(0, cols, 2):
                low_byte = int(matrix_hex[i, j])      # 첫번째 원소 -> 하위 8비트
                high_byte = int(matrix_hex[i, j+1])   # 두번째 원소 -> 상위 8비트
                combined = (high_byte << input_width) | low_byte
                line_idx += 1
                if line_idx < total_lines:
                    f.write(f"{combined:04X},\n")
                else:
                    f.write(f"{combined:04X};\n")
    
    print(f"{filename} 생성 완료! ({total_lines}줄)")

# matrix_a와 matrix_b_trans를 COE 파일로 저장
save_matrix_to_coe(matrix_a, "matrix_a.coe", input_width)
save_matrix_to_coe(vector_b.reshape(-1, 2), "vector_b.coe", input_width)

# 검증 출력
print("\nMatrix A COE (first 4 lines):")
matrix_a_hex = matrix_a.astype(np.uint8)
for i in range(4):
    low = int(matrix_a_hex[0, i*2])
    high = int(matrix_a_hex[0, i*2+1])
    print(f"  A[0,{i*2}]={low:02X}, A[0,{i*2+1}]={high:02X} -> {(high<<input_width)|low:04X}")

print("\nVector B COE (first 4 lines):")
vector_b_hex = vector_b.astype(np.uint8)
for i in range(4):
    low = int(vector_b_hex[i*2])
    high = int(vector_b_hex[i*2+1])
    print(f"  B[{i*2}]={low:02X}, B[{i*2+1}]={high:02X} -> {(high<<input_width)|low:04X}")


matrix_a.coe 생성 완료! (512줄)
vector_b.coe 생성 완료! (16줄)

Matrix A COE (first 4 lines):
  A[0,0]=5B, A[0,1]=6A -> 6A5B
  A[0,2]=31, A[0,3]=90 -> 9031
  A[0,4]=C5, A[0,5]=CF -> CFC5
  A[0,6]=CC, A[0,7]=4F -> 4FCC

Vector B COE (first 4 lines):
  B[0]=5D, B[1]=C7 -> C75D
  B[2]=26, B[3]=CE -> CE26
  B[4]=47, B[5]=3E -> 3E47
  B[6]=7C, B[7]=B4 -> B47C


In [9]:
# C 벡터 계산 (A * b 행렬-벡터 곱)
if (result_width==32):
    d_type_r=np.int32

# int32로 변환 후 행렬-벡터 곱 (오버플로우 방지)
matrix_c = np.matmul(matrix_a.astype(d_type_r), vector_b.astype(d_type_r))

# 32비트 2의 보수로 변환하는 함수
def to_unsigned_hex(val):
    val = int(val)
    if val < 0:
        return val & 0xFFFFFFFF
    return val

print("Vector C (first 8 elements):")
print(" ".join(f"{to_unsigned_hex(matrix_c[i]):08X}" for i in range(8)))

print(f"\nVector C shape: {matrix_c.shape}")
print(f"Vector C min: {matrix_c.min()}, max: {matrix_c.max()}")

# vector_c.coe 생성 (32비트, 한 줄에 한 원소)
with open("vector_c.coe", 'w') as f:
    f.write("memory_initialization_radix=16;\n")
    f.write("memory_initialization_vector=\n")
    total_lines = mat_size
    for i in range(total_lines):
        val = to_unsigned_hex(matrix_c[i])
        if i < total_lines - 1:
            f.write(f"{val:08X},\n")
        else:
            f.write(f"{val:08X};\n")

print(f"\nvector_c.coe 파일 생성 완료! ({total_lines}줄)")

# 검증 출력
print("\nVector C COE (first 4 elements):")
for i in range(4):
    val = int(matrix_c[i])
    hex_val = to_unsigned_hex(val)
    print(f"  C[{i}] = {val} -> {hex_val:08X}")

Vector C (first 8 elements):
000036D9 FFFFB23B 0000849A FFFFE273 FFFF81AE 00000828 00001934 000044B6

Vector C shape: (32,)
Vector C min: -51763, max: 49540

vector_c.coe 파일 생성 완료! (32줄)

Vector C COE (first 4 elements):
  C[0] = 14041 -> 000036D9
  C[1] = -19909 -> FFFFB23B
  C[2] = 33946 -> 0000849A
  C[3] = -7565 -> FFFFE273


In [10]:
# sparse_a와 matrix_b의 행렬 곱 계산
# 오버플로우 방지를 위해 int32로 변환
sparse_c = np.matmul(sparse_a.astype(d_type_r), vector_b.astype(d_type_r))

print("=== Sparse A × Matrix B ===")
print(f"Sparse C shape: {sparse_c.shape}")
print(f"Sparse C min: {sparse_c.min()}, max: {sparse_c.max()}")

# 32비트 2의 보수로 변환하는 함수
def to_unsigned_hex(val):
    val = int(val)
    if val < 0:
        return val & 0xFFFFFFFF
    return val

print("\nSparse C (first 4x4, Decimal):")
print(sparse_c[:4, :4])

print("\nSparse C (first 4x4, Hex):")
for i in range(4):
    print(" ".join(f"{to_unsigned_hex(sparse_c[i,j]):08X}" for j in range(4)))

# sparse_c.coe 생성
with open("sparse_c.coe", 'w') as f:
    f.write("memory_initialization_radix=16;\n")
    f.write("memory_initialization_vector=\n")
    total_lines = mat_size * mat_size
    line_idx = 0
    for i in range(mat_size):
        for j in range(mat_size):
            val = to_unsigned_hex(sparse_c[i, j])
            line_idx += 1
            if line_idx < total_lines:
                f.write(f"{val:08X},\n")
            else:
                f.write(f"{val:08X};\n")

print(f"\nsparse_c.coe 파일 생성 완료! ({total_lines}줄)")

=== Sparse A × Matrix B ===
Sparse C shape: (32,)
Sparse C min: -26865, max: 30340

Sparse C (first 4x4, Decimal):


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [ ]:
def csr_matvec(csr_values, csr_col_indices, csr_row_ptr, vector):
    """
    CSR 포맷의 희소 행렬 A와 벡터 b의 곱셈 (A × b)
    
    Parameters:
        csr_values: A의 0이 아닌 값들
        csr_col_indices: 각 값의 열 인덱스
        csr_row_ptr: 각 행의 시작 위치
        vector: 벡터 b (N,)
    
    Returns:
        result: A × b 결과 벡터 (M,)
    """
    num_rows = len(csr_row_ptr) - 1
    result = np.zeros(num_rows, dtype=np.int32)
    
    for i in range(num_rows):
        row_start = csr_row_ptr[i]
        row_end = csr_row_ptr[i + 1]
        
        for idx in range(row_start, row_end):
            a_val = csr_values[idx]         # A[i, k]의 값
            k = csr_col_indices[idx]        # 열 인덱스 k
            result[i] += int(a_val) * int(vector[k])
    
    return result

# CSR 형태로 행렬 곱 계산
sparse_c_csr = csr_matvec(csr_values, csr_col_indices, csr_row_ptr, vector_b)

print("=== CSR-based Matrix Multiplication ===")
print(f"Result shape: {sparse_c_csr.shape}")
print(f"Result min: {sparse_c_csr.min()}, max: {sparse_c_csr.max()}")

print("\nSparse C via CSR (first 4, Decimal):")
print(sparse_c_csr[:4])

print("\nSparse C via CSR (first 4, Hex):")
for i in range(4):
    print(" ".join(f"{to_unsigned_hex(sparse_c_csr[i,j]):08X}" for j in range(4)))

# 검증: numpy matmul 결과와 비교
print(f"\nVerification (CSR vs numpy): {np.array_equal(sparse_c, sparse_c_csr)}")

# 연산 횟수 비교
dense_ops = mat_size * mat_size * mat_size  # 일반 행렬곱
sparse_ops = len(csr_values) * mat_size     # CSR 기반 곱셈
print(f"\nDense multiplication ops: {dense_ops}")
print(f"CSR-based multiplication ops: {sparse_ops}")
print(f"Speedup ratio: {dense_ops / sparse_ops:.2f}x")

=== CSR-based Matrix Multiplication ===
Result shape: (32, 32)
Result min: -64594, max: 95870

Sparse C via CSR (first 4x4, Decimal):
[[  6787  13628  -5752 -17275]
 [-20443  28914   2441  19185]
 [-15925 -27356 -21643 -38593]
 [-10826 -10906  20311  15346]]

Sparse C via CSR (first 4x4, Hex):
00001A83 0000353C FFFFE988 FFFFBC85
FFFFB025 000070F2 00000989 00004AF1
FFFFC1CB FFFF9524 FFFFAB75 FFFF693F
FFFFD5B6 FFFFD566 00004F57 00003BF2

Verification (CSR vs numpy): True

Dense multiplication ops: 32768
CSR-based multiplication ops: 16320
Speedup ratio: 2.01x
